# YOLOv12 for Diabetic Foot Ulcer (DFU) Object Detection
This notebook trains a **YOLOv12** model for detecting and localizing diabetic foot ulcers in images.
Designed to run on **Kaggle GPU** environment.

**Training Configuration:**
- **Model**: YOLOv12 (pretrained on COCO)
- **Epochs**: 100
- **Image Size**: 640×640
- **Batch Size**: 16
- **Optimizer**: AdamW (auto)
- **Dataset**: Roboflow Diabetic Ulcers (YOLOv12 format)
- **Augmentation**: YOLO built-in (mosaic, mixup, hsv, flip, scale)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# KAGGLE SETUP — Clone repo & install dependencies
# Run this cell FIRST on Kaggle to pull the project code.
# ══════════════════════════════════════════════════════════════════════════════
import os

REPO_URL    = 'https://github.com/csstudentkaum/KHOTAA.git'
BRANCH      = 'main'
REPO_DIR    = '/kaggle/working/KHOTAA'
WORKING_DIR = os.path.join(REPO_DIR, 'models', 'detection')

# Clone the repo (skip if already cloned)
if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}
    print(f'Cloned {BRANCH} branch -> {REPO_DIR}')
else:
    print(f'Repo already exists at {REPO_DIR}')

os.chdir(WORKING_DIR)
print(f'Working directory: {os.getcwd()}')

# Install dependencies
!pip install roboflow -q
!pip install ultralytics -q
print('Kaggle setup complete')

## 1. Imports & Configuration

In [ ]:
# ── Standard Library ──────────────────────────────────────────────────────────
import sys
import os
import random
import warnings
import json
import glob
import shutil
warnings.filterwarnings('ignore')

# ── Numerical & Visualization ─────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image
from collections import Counter
from datetime import datetime
from pathlib import Path

# ── PyTorch (for GPU check) ───────────────────────────────────────────────────
import torch

# ── YOLO ──────────────────────────────────────────────────────────────────────
from ultralytics import YOLO

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Hyperparameters ───────────────────────────────────────────────────────────
IMAGE_SIZE   = 640
BATCH_SIZE   = 16
EPOCHS       = 100
PATIENCE     = 20
CONF_THRESH  = 0.25
IOU_THRESH   = 0.5

# ── Roboflow Dataset ──────────────────────────────────────────────────────────
from roboflow import Roboflow
rf = Roboflow(api_key="2n2g7aUdK6UhuWNJRxNE")
project = rf.workspace("ssk-r6ppk").project("diabetic_ulcers")
version = project.version(3)
dataset = version.download("yolov12")
DATASET_PATH = dataset.location

# ── Directories ───────────────────────────────────────────────────────────────
MODEL_NAME  = 'YOLOv12'
RESULTS_DIR = 'results'

print('Imports complete')
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
print(f'Image size      : {IMAGE_SIZE}')
print(f'Batch size      : {BATCH_SIZE}')
print(f'Epochs          : {EPOCHS}')
print(f'Dataset path    : {DATASET_PATH}')

## 2. Explore Dataset Structure

In [ ]:
# ── Verify dataset directory structure ────────────────────────────────────────
print('Dataset root contents:')
for entry in sorted(os.listdir(DATASET_PATH)):
    full = os.path.join(DATASET_PATH, entry)
    kind = 'DIR' if os.path.isdir(full) else 'FILE'
    print(f'  [{kind}] {entry}')

# ── Read data.yaml ────────────────────────────────────────────────────────────
import yaml
data_yaml_path = os.path.join(DATASET_PATH, 'data.yaml')
with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

print(f'\ndata.yaml contents:')
for k, v in data_config.items():
    print(f'  {k}: {v}')

class_names = data_config.get('names', [])
num_classes = data_config.get('nc', len(class_names))
print(f'\nClasses ({num_classes}): {class_names}')

# ── Count images and labels per split ─────────────────────────────────────────
for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(DATASET_PATH, split, 'images')
    lbl_dir = os.path.join(DATASET_PATH, split, 'labels')
    if os.path.exists(img_dir):
        n_imgs = len(glob.glob(os.path.join(img_dir, '*')))
        n_lbls = len(glob.glob(os.path.join(lbl_dir, '*.txt'))) if os.path.exists(lbl_dir) else 0
        print(f'\n{split.upper()} split:')
        print(f'  Images: {n_imgs}')
        print(f'  Labels: {n_lbls}')
    else:
        print(f'\n{split.upper()} split: NOT FOUND')

In [ ]:
# ── Class Distribution per Split ──────────────────────────────────────────────
PALETTE = ["#3D6A99", "#85B1D2", "#64ADB3", "#2A4D6E", "#A0522D", "#CD853F"]

split_counts = {}
for split in ['train', 'valid', 'test']:
    lbl_dir = os.path.join(DATASET_PATH, split, 'labels')
    if not os.path.exists(lbl_dir):
        continue
    class_counter = Counter()
    for lbl_file in glob.glob(os.path.join(lbl_dir, '*.txt')):
        with open(lbl_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(parts[0])
                    class_counter[cls_id] += 1
    split_counts[split] = class_counter

# Plot
n_splits = len(split_counts)
fig, axes = plt.subplots(1, n_splits, figsize=(6 * n_splits, 5))
if n_splits == 1:
    axes = [axes]

for ax, (split, counter) in zip(axes, split_counts.items()):
    cls_ids = sorted(counter.keys())
    counts  = [counter[c] for c in cls_ids]
    labels  = [class_names[c] if c < len(class_names) else f'Class {c}' for c in cls_ids]
    colors  = [PALETTE[i % len(PALETTE)] for i in range(len(cls_ids))]
    bars = ax.bar(labels, counts, color=colors, edgecolor='white')
    ax.set_title(f'{split.upper()} - Annotation Distribution', fontsize=13, fontweight='bold')
    ax.set_xlabel('Class', fontsize=11)
    ax.set_ylabel('Number of Annotations', fontsize=11)
    for bar, cnt in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                str(cnt), ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)
    total = sum(counts)
    ax.set_title(f'{split.upper()} ({total} annotations)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

# Print summary
print('\nAnnotation Summary:')
for split, counter in split_counts.items():
    total = sum(counter.values())
    print(f'  {split}: {total} annotations')
    for cls_id in sorted(counter.keys()):
        name = class_names[cls_id] if cls_id < len(class_names) else f'Class {cls_id}'
        print(f'    {name}: {counter[cls_id]}')

In [ ]:
# ── Display Sample Images with Bounding Boxes ─────────────────────────────────
BOX_COLORS = ['#FF4444', '#44FF44', '#4444FF', '#FFFF44', '#FF44FF', '#44FFFF']

def show_yolo_samples(dataset_path, split='train', n_samples=6, class_names=None):
    """Display sample images with their YOLO bounding box annotations."""
    img_dir = os.path.join(dataset_path, split, 'images')
    lbl_dir = os.path.join(dataset_path, split, 'labels')
    
    img_files = sorted(glob.glob(os.path.join(img_dir, '*')))
    # Pick samples that have annotations
    samples = []
    for img_path in img_files:
        stem = Path(img_path).stem
        lbl_path = os.path.join(lbl_dir, stem + '.txt')
        if os.path.exists(lbl_path) and os.path.getsize(lbl_path) > 0:
            samples.append((img_path, lbl_path))
        if len(samples) >= n_samples:
            break
    
    n_cols = min(3, len(samples))
    n_rows = (len(samples) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 6 * n_rows))
    if n_rows == 1 and n_cols == 1:
        axes = np.array([axes])
    axes = np.array(axes).flatten()
    
    for idx, (img_path, lbl_path) in enumerate(samples):
        img = Image.open(img_path).convert('RGB')
        w, h = img.size
        ax = axes[idx]
        ax.imshow(img)
        
        with open(lbl_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(parts[0])
                    cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                    x1 = (cx - bw/2) * w
                    y1 = (cy - bh/2) * h
                    box_w = bw * w
                    box_h = bh * h
                    color = BOX_COLORS[cls_id % len(BOX_COLORS)]
                    rect = patches.Rectangle((x1, y1), box_w, box_h,
                                             linewidth=2, edgecolor=color, facecolor='none')
                    ax.add_patch(rect)
                    label = class_names[cls_id] if class_names and cls_id < len(class_names) else f'Class {cls_id}'
                    ax.text(x1, y1 - 5, label, color=color, fontsize=10,
                            fontweight='bold', backgroundcolor='black')
        
        ax.set_title(Path(img_path).name, fontsize=10)
        ax.axis('off')
    
    for idx in range(len(samples), len(axes)):
        axes[idx].axis('off')
    
    fig.suptitle(f'{split.upper()} Set - Sample Images with Bounding Boxes',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

show_yolo_samples(DATASET_PATH, split='train', n_samples=6, class_names=class_names)

## 3. Bounding Box Size Analysis

In [ ]:
# ── Bounding Box Size Distribution ────────────────────────────────────────────
box_widths  = []
box_heights = []
box_areas   = []
box_classes = []

train_lbl_dir = os.path.join(DATASET_PATH, 'train', 'labels')
for lbl_file in glob.glob(os.path.join(train_lbl_dir, '*.txt')):
    with open(lbl_file, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                cls_id = int(parts[0])
                bw, bh = float(parts[3]), float(parts[4])
                box_widths.append(bw)
                box_heights.append(bh)
                box_areas.append(bw * bh)
                box_classes.append(cls_id)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'{MODEL_NAME} - Bounding Box Analysis (Training Set)',
             fontsize=14, fontweight='bold')

axes[0].hist(box_widths, bins=30, color=PALETTE[0], edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Normalized Width', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_title('BBox Width Distribution', fontweight='bold')
axes[0].axvline(np.mean(box_widths), color='red', linestyle='--', label=f'Mean: {np.mean(box_widths):.3f}')
axes[0].legend()

axes[1].hist(box_heights, bins=30, color=PALETTE[1], edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Normalized Height', fontsize=11)
axes[1].set_ylabel('Count', fontsize=11)
axes[1].set_title('BBox Height Distribution', fontweight='bold')
axes[1].axvline(np.mean(box_heights), color='red', linestyle='--', label=f'Mean: {np.mean(box_heights):.3f}')
axes[1].legend()

axes[2].scatter(box_widths, box_heights, c=[PALETTE[c % len(PALETTE)] for c in box_classes],
                alpha=0.5, s=20, edgecolors='none')
axes[2].set_xlabel('Normalized Width', fontsize=11)
axes[2].set_ylabel('Normalized Height', fontsize=11)
axes[2].set_title('BBox Width vs Height', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'\nBounding Box Statistics (Training Set):')
print(f'  Total annotations: {len(box_widths)}')
print(f'  Width  - Mean: {np.mean(box_widths):.4f}, Std: {np.std(box_widths):.4f}')
print(f'  Height - Mean: {np.mean(box_heights):.4f}, Std: {np.std(box_heights):.4f}')
print(f'  Area   - Mean: {np.mean(box_areas):.4f}, Std: {np.std(box_areas):.4f}')

## 4. Load YOLOv12 Model

In [ ]:
# ── Load pretrained YOLOv12 model ─────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# YOLOv12 medium model (pretrained on COCO)
model = YOLO('yolo12m.pt')

print(f'\nYOLOv12 model loaded')
print(f'Model type    : {model.type}')
print(f'Task          : {model.task}')
print(f'Input size    : {IMAGE_SIZE}x{IMAGE_SIZE}')
print(f'Output classes: {num_classes} (will be fine-tuned)')

## 5. Training

Training configuration:
- **Model**: YOLOv12m (medium) pretrained on COCO
- **Optimizer**: AdamW (auto-selected by Ultralytics)
- **Scheduler**: Cosine annealing with warmup
- **Epochs**: 100 (with early stopping patience=20)
- **Augmentation**: YOLO built-in (mosaic, mixup, hsv, flip, scale)
- **Image Size**: 640×640

In [ ]:
# ── Train YOLOv12 ─────────────────────────────────────────────────────────────
results = model.train(
    data=os.path.join(DATASET_PATH, 'data.yaml'),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    patience=PATIENCE,
    save=True,
    save_period=-1,
    device=device,
    workers=2,
    seed=SEED,
    verbose=True,
    plots=True,
    # Augmentation (YOLO defaults)
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    flipud=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.0,
    # Project settings
    project='results',
    name='yolov12',
    exist_ok=True,
)

print('\nTraining complete!')
print(f'Best model saved at: {model.trainer.best}')
print(f'Last model saved at: {model.trainer.last}')

## 6. Training Curves & Metrics

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
DPI = 300
FONT = {'title': 14, 'label': 12, 'legend': 10, 'tick': 10}

train_results_dir = str(model.trainer.save_dir)
print(f'Results directory: {train_results_dir}')

# ── Read training results CSV ─────────────────────────────────────────────────
import pandas as pd
results_csv = os.path.join(train_results_dir, 'results.csv')
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()
print(f'Training epochs logged: {len(df)}')
print(f'Columns: {list(df.columns)}')

# ══════════════════════════════════════════════════════════════════════════════
# 1. LOSS CURVES
# ══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'{MODEL_NAME} - Training Loss Curves', fontsize=FONT['title']+2, fontweight='bold')

loss_pairs = [
    ('train/box_loss', 'val/box_loss', 'Box Loss'),
    ('train/cls_loss', 'val/cls_loss', 'Classification Loss'),
    ('train/dfl_loss', 'val/dfl_loss', 'DFL Loss'),
]

for ax, (train_col, val_col, title) in zip(axes, loss_pairs):
    if train_col in df.columns and val_col in df.columns:
        epochs = range(1, len(df) + 1)
        ax.plot(epochs, df[train_col], 'b-', label='Train', lw=2)
        ax.plot(epochs, df[val_col],   'r-', label='Val',   lw=2)
        best_epoch = df[val_col].idxmin() + 1
        ax.axvline(x=best_epoch, color='g', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
        ax.set_xlabel('Epoch', fontsize=FONT['label'])
        ax.set_ylabel('Loss', fontsize=FONT['label'])
        ax.set_title(title, fontsize=FONT['title'], fontweight='bold')
        ax.legend(fontsize=FONT['legend'])
        ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(train_results_dir, f'{MODEL_NAME.lower()}_loss_curves.png'),
            dpi=DPI, bbox_inches='tight')
plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# 2. METRIC CURVES (mAP, Precision, Recall)
# ══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'{MODEL_NAME} - Validation Metrics', fontsize=FONT['title']+2, fontweight='bold')

metric_cols = [
    ('metrics/mAP50(B)', 'mAP@50', '#3D6A99'),
    ('metrics/mAP50-95(B)', 'mAP@50-95', '#64ADB3'),
    ('metrics/precision(B)', 'Precision', '#85B1D2'),
]

# Also plot recall if available
recall_col = 'metrics/recall(B)'

for ax, (col, title, color) in zip(axes, metric_cols):
    if col in df.columns:
        epochs = range(1, len(df) + 1)
        ax.plot(epochs, df[col], '-', color=color, lw=2.5, label=title)
        if col == 'metrics/mAP50(B)' and recall_col in df.columns:
            ax.plot(epochs, df[recall_col], '-', color='#2A4D6E', lw=2, label='Recall', alpha=0.7)
        best_val = df[col].max()
        best_ep  = df[col].idxmax() + 1
        ax.axhline(y=best_val, color='red', linestyle='--', alpha=0.4,
                   label=f'Best: {best_val:.4f} (ep {best_ep})')
        ax.set_xlabel('Epoch', fontsize=FONT['label'])
        ax.set_ylabel(title, fontsize=FONT['label'])
        ax.set_title(title, fontsize=FONT['title'], fontweight='bold')
        ax.legend(fontsize=FONT['legend'])
        ax.grid(alpha=0.3)
        ax.set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig(os.path.join(train_results_dir, f'{MODEL_NAME.lower()}_metric_curves.png'),
            dpi=DPI, bbox_inches='tight')
plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# 3. DISPLAY YOLO AUTO-GENERATED PLOTS
# ══════════════════════════════════════════════════════════════════════════════
auto_plots = [
    'confusion_matrix.png',
    'confusion_matrix_normalized.png',
    'F1_curve.png',
    'PR_curve.png',
    'P_curve.png',
    'R_curve.png',
    'results.png',
]

for plot_name in auto_plots:
    plot_path = os.path.join(train_results_dir, plot_name)
    if os.path.exists(plot_path):
        img = Image.open(plot_path)
        fig, ax = plt.subplots(figsize=(12, 8))
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(plot_name.replace('.png', '').replace('_', ' ').title(),
                     fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
    else:
        print(f'Plot not found: {plot_name}')

# ══════════════════════════════════════════════════════════════════════════════
# TRAINING SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print(f'{MODEL_NAME} TRAINING SUMMARY')
print(f"{'='*60}")
print(f'Total epochs trained : {len(df)}')
if 'metrics/mAP50(B)' in df.columns:
    print(f'Best mAP@50          : {df["metrics/mAP50(B)"].max():.4f} (epoch {df["metrics/mAP50(B)"].idxmax()+1})')
if 'metrics/mAP50-95(B)' in df.columns:
    print(f'Best mAP@50-95       : {df["metrics/mAP50-95(B)"].max():.4f} (epoch {df["metrics/mAP50-95(B)"].idxmax()+1})')
if 'metrics/precision(B)' in df.columns:
    print(f'Best Precision       : {df["metrics/precision(B)"].max():.4f}')
if recall_col in df.columns:
    print(f'Best Recall          : {df[recall_col].max():.4f}')
print(f"{'='*60}")

## 7. Evaluate on Validation Set

In [ ]:
# ── Load best model and evaluate on validation set ────────────────────────────
best_model_path = os.path.join(train_results_dir, 'weights', 'best.pt')
best_model = YOLO(best_model_path)
print(f'Loaded best model from: {best_model_path}')

# Run validation
val_results = best_model.val(
    data=os.path.join(DATASET_PATH, 'data.yaml'),
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    conf=CONF_THRESH,
    iou=IOU_THRESH,
    device=device,
    plots=True,
    save_json=True,
    project='results',
    name='yolov12_val',
    exist_ok=True,
)

# Print detailed results
print(f"\n{'='*60}")
print(f'{MODEL_NAME} VALIDATION RESULTS')
print(f"{'='*60}")
print(f'mAP@50       : {val_results.box.map50:.4f}')
print(f'mAP@50-95    : {val_results.box.map:.4f}')
print(f'Precision    : {val_results.box.mp:.4f}')
print(f'Recall       : {val_results.box.mr:.4f}')

# Per-class results
print(f'\nPer-Class Results:')
for i, name in enumerate(class_names):
    if i < len(val_results.box.ap50):
        print(f'  {name}:')
        print(f'    AP@50    : {val_results.box.ap50[i]:.4f}')
        print(f'    AP@50-95 : {val_results.box.ap[i]:.4f}')
print(f"{'='*60}")

## 8. Inference on Sample Images

In [ ]:
# ── Run inference on sample validation images ─────────────────────────────────
val_img_dir = os.path.join(DATASET_PATH, 'valid', 'images')
val_images = sorted(glob.glob(os.path.join(val_img_dir, '*')))[:8]

n_cols = 4
n_rows = (len(val_images) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
axes = np.array(axes).flatten()

for idx, img_path in enumerate(val_images):
    # Run inference
    pred_results = best_model.predict(
        source=img_path,
        conf=CONF_THRESH,
        iou=IOU_THRESH,
        imgsz=IMAGE_SIZE,
        device=device,
        verbose=False
    )
    
    # Plot result
    result = pred_results[0]
    annotated = result.plot()
    annotated = annotated[:, :, ::-1]  # BGR -> RGB
    
    ax = axes[idx]
    ax.imshow(annotated)
    ax.set_title(f'{Path(img_path).name}\n({len(result.boxes)} detections)', fontsize=9)
    ax.axis('off')

for idx in range(len(val_images), len(axes)):
    axes[idx].axis('off')

fig.suptitle(f'{MODEL_NAME} - Inference Results (Validation Set)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(train_results_dir, f'{MODEL_NAME.lower()}_inference_samples.png'),
            dpi=DPI, bbox_inches='tight')
plt.show()

print(f'Inference samples saved')

## 9. Generate Comprehensive Metrics JSON

This cell generates a standalone comprehensive metrics JSON file from the trained YOLOv12 model.
It includes all detection metrics (mAP, precision, recall, per-class AP) for cross-model comparison.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# GENERATE COMPREHENSIVE METRICS JSON
# ══════════════════════════════════════════════════════════════════════════════
print('='*80)
print('GENERATING COMPREHENSIVE METRICS')
print('='*80)

# ── Read training CSV for history ─────────────────────────────────────────────
import pandas as pd
results_csv = os.path.join(train_results_dir, 'results.csv')
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

# ── Per-class metrics ─────────────────────────────────────────────────────────
per_class_metrics = {}
for i, name in enumerate(class_names):
    metrics = {'class_id': i}
    if i < len(val_results.box.ap50):
        metrics['ap50']    = float(val_results.box.ap50[i])
        metrics['ap50_95'] = float(val_results.box.ap[i])
    per_class_metrics[name] = metrics

# ── Training history ──────────────────────────────────────────────────────────
training_history = {}
for col in df.columns:
    if col.strip() != 'epoch':
        training_history[col] = df[col].tolist()

# ── Build comprehensive results ──────────────────────────────────────────────
comprehensive_results = {
    'model_info': {
        'model_name':        MODEL_NAME,
        'architecture':      'YOLOv12m (medium)',
        'task':              'object_detection',
        'pretrained':        True,
        'pretrained_weights': 'COCO',
        'input_size':        [IMAGE_SIZE, IMAGE_SIZE],
        'num_classes':       num_classes,
        'class_names':       class_names
    },
    'training_config': {
        'optimizer':             'AdamW (auto)',
        'batch_size':            BATCH_SIZE,
        'num_epochs':            EPOCHS,
        'actual_epochs':         len(df),
        'early_stopping_patience': PATIENCE,
        'image_size':            IMAGE_SIZE,
        'confidence_threshold':  CONF_THRESH,
        'iou_threshold':         IOU_THRESH,
        'augmentation': {
            'mosaic':    1.0,
            'mixup':     0.0,
            'hsv_h':     0.015,
            'hsv_s':     0.7,
            'hsv_v':     0.4,
            'flipud':    0.5,
            'fliplr':    0.5,
            'scale':     0.5,
            'translate': 0.1
        }
    },
    'dataset_info': {
        'dataset_name': 'Diabetic Ulcers Detection',
        'source':       'Roboflow (ssk-r6ppk / diabetic_ulcers v3)',
        'format':       'YOLOv12',
        'splits': {}
    },
    'validation_results': {
        'mAP50':     float(val_results.box.map50),
        'mAP50_95':  float(val_results.box.map),
        'precision':  float(val_results.box.mp),
        'recall':     float(val_results.box.mr),
        'per_class_metrics': per_class_metrics
    },
    'training_history': {
        'total_epochs': len(df),
        'best_mAP50':     float(df['metrics/mAP50(B)'].max()) if 'metrics/mAP50(B)' in df.columns else None,
        'best_mAP50_epoch': int(df['metrics/mAP50(B)'].idxmax() + 1) if 'metrics/mAP50(B)' in df.columns else None,
        'best_mAP50_95':  float(df['metrics/mAP50-95(B)'].max()) if 'metrics/mAP50-95(B)' in df.columns else None,
        'best_mAP50_95_epoch': int(df['metrics/mAP50-95(B)'].idxmax() + 1) if 'metrics/mAP50-95(B)' in df.columns else None,
        'final_train_box_loss':  float(df['train/box_loss'].iloc[-1]) if 'train/box_loss' in df.columns else None,
        'final_train_cls_loss':  float(df['train/cls_loss'].iloc[-1]) if 'train/cls_loss' in df.columns else None,
        'final_val_box_loss':    float(df['val/box_loss'].iloc[-1]) if 'val/box_loss' in df.columns else None,
        'final_val_cls_loss':    float(df['val/cls_loss'].iloc[-1]) if 'val/cls_loss' in df.columns else None,
        'curves': training_history
    },
    'metadata': {
        'timestamp':            datetime.now().isoformat(),
        'framework':            'Ultralytics (PyTorch)',
        'pytorch_version':      torch.__version__,
        'cuda_available':       torch.cuda.is_available(),
        'experiment_name':      'yolov12_dfu_detection',
        'best_model_path':      best_model_path,
        'results_directory':    train_results_dir,
        'generated_standalone': True
    }
}

# Add dataset split counts
for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(DATASET_PATH, split, 'images')
    if os.path.exists(img_dir):
        n_imgs = len(glob.glob(os.path.join(img_dir, '*')))
        comprehensive_results['dataset_info']['splits'][split] = {
            'num_images': n_imgs,
            'annotations': dict(split_counts.get(split, {}))
        }

# ── Save JSON ─────────────────────────────────────────────────────────────────
os.makedirs(train_results_dir, exist_ok=True)
comprehensive_json_path = os.path.join(train_results_dir, f'{MODEL_NAME.lower()}_comprehensive_metrics.json')
with open(comprehensive_json_path, 'w') as f:
    json.dump(comprehensive_results, f, indent=2)

# ══════════════════════════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
print(f'\n{"="*80}')
print('COMPREHENSIVE METRICS SUMMARY')
print(f'{"="*80}')
print(f'\nMODEL: {MODEL_NAME}')
print(f'Task : Object Detection')
print(f'\nVALIDATION RESULTS:')
print(f'  mAP@50     : {val_results.box.map50:.4f}')
print(f'  mAP@50-95  : {val_results.box.map:.4f}')
print(f'  Precision  : {val_results.box.mp:.4f}')
print(f'  Recall     : {val_results.box.mr:.4f}')
print(f'\nTRAINING:')
print(f'  Total epochs : {len(df)}')
if 'metrics/mAP50(B)' in df.columns:
    print(f'  Best mAP@50  : {df["metrics/mAP50(B)"].max():.4f} (epoch {df["metrics/mAP50(B)"].idxmax()+1})')
print(f'\nPER-CLASS RESULTS:')
for name, m in per_class_metrics.items():
    print(f'  {name}:')
    if 'ap50' in m:
        print(f'    AP@50    : {m["ap50"]:.4f}')
    if 'ap50_95' in m:
        print(f'    AP@50-95 : {m["ap50_95"]:.4f}')
print(f'\n{"="*80}')
print(f'Saved comprehensive metrics to:')
print(f'  {os.path.abspath(comprehensive_json_path)}')
file_size = os.path.getsize(comprehensive_json_path) / 1024
print(f'  File size: {file_size:.2f} KB')
print(f'{"="*80}')
print('COMPLETE')
print(f'{"="*80}')